# Capstone Track A — Domain RAG Assistant
**Day 2 Afternoon | ~3 hours | Colab CPU | OpenAI API key required**

---

## What You Are Building
A RAG-powered chat assistant over documents from a domain **you choose**. You will load documents, chunk and embed them, write a domain-specific grounding prompt, and ship a streaming Gradio app with a public URL.

**Your job in this template:**
- [ ] Choose a domain and load real documents (Step 1)
- [ ] Choose chunk size — justify your decision (Step 2)
- [ ] Write the grounding system prompt for your domain (Step 3)
- [ ] Test retrieval quality and iterate (Step 4)
- [ ] Customise the Gradio UI with your domain's example questions (Step 5)

Everything else (plumbing, Gradio shell, RAGAS eval) is provided.

> **Track B (fine-tuning)?** Open `capstone_track_b.ipynb` instead — it requires a T4 GPU.

In [ ]:
%%capture
!pip install sentence-transformers chromadb langchain langchain-community langchain-openai openai gradio pypdf bm25s ragas datasets
print('Done')


In [ ]:
# Configuration — works in Colab Secrets or local environment variables
import os

def get_secret(name: str, *, required: bool = True):
    value = os.environ.get(name)
    try:
        from google.colab import userdata
        value = userdata.get(name) or value
    except Exception:
        pass
    if required and not value:
        raise ValueError(
            f"Missing {name}. In Colab, open the key icon in the left sidebar, "
            f"add a secret named {name}, paste the value, and enable Notebook access."
        )
    return value

OPENAI_API_KEY  = get_secret('OPENAI_API_KEY')
OPENAI_BASE_URL = 'https://api.openai.com/v1'
DEFAULT_MODEL   = 'gpt-4o-mini'
EMBED_MODEL     = 'all-MiniLM-L6-v2'

print(f'Config loaded — OPENAI_API_KEY starts with {OPENAI_API_KEY[:8]}...')


---
## Step 1 — Load Your Documents

Choose **one** option below and run it. The other two stay commented out.

| Option | Best for | Effort |
|--------|----------|--------|
| **A — Inline text** | Quick start, guaranteed to work | Low |
| **B — Upload PDFs** | Real documents you own | Medium |
| **C — Web URLs** | Public documentation, Wikipedia | Low |

> **Tip:** Start with Option A to make sure the pipeline works end-to-end, then swap in your real documents.

In [ ]:
# ── Option A: Inline text (DEFAULT — always works, no file upload needed) ────
# Replace these placeholders with content from your chosen domain.
# Aim for at least 3 topics, 100+ words each.

# TODO: replace with your domain's content
raw_documents = {
    'topic_1': '''
    Paste your first document or topic summary here.
    Make it at least a paragraph — short snippets produce poor retrieval.
    ''',
    'topic_2': '''
    Paste your second document here.
    ''',
    'topic_3': '''
    Paste your third document here.
    ''',
}

# Convert to the format the pipeline expects
all_text_chunks_raw = list(raw_documents.values())
source_labels = list(raw_documents.keys())
print(f'✅ Loaded {len(all_text_chunks_raw)} documents (inline)')

In [ ]:
# ── Option B: Upload PDFs ────────────────────────────────────────────────────
# Uncomment this cell and run it to upload PDF files from your computer.
# A file-picker dialog will appear. Select one or more PDFs.
# They will be saved to a 'pdfs/' folder in the Colab runtime.

# import os
# from google.colab import files
# from langchain_community.document_loaders import PyPDFLoader

# # Create the pdfs folder
# os.makedirs('pdfs', exist_ok=True)

# # Upload — a file picker dialog will open
# print('Select your PDF files to upload:')
# uploaded = files.upload()

# # Move files into pdfs/
# for fname in uploaded:
#     os.rename(fname, f'pdfs/{fname}')
#     print(f'  ✅ {fname} → pdfs/{fname}')

# # Load all PDFs from pdfs/
# all_docs = []
# for fname in sorted(os.listdir('pdfs')):
#     if fname.lower().endswith('.pdf'):
#         pages = PyPDFLoader(f'pdfs/{fname}').load()
#         all_docs.extend(pages)
#         print(f'  📄 {fname}: {len(pages)} pages')

# all_text_chunks_raw = [doc.page_content for doc in all_docs]
# source_labels = [doc.metadata.get('source', 'pdf') for doc in all_docs]
# print(f'\n✅ Loaded {len(all_text_chunks_raw)} pages from {len(uploaded)} PDFs')

print('Option B is commented out. Uncomment the block above to use PDF upload.')

In [ ]:
# ── Option C: Load from Web URLs ─────────────────────────────────────────────
# Uncomment and edit the URLs list. Runs automatically — no upload needed.

# from langchain_community.document_loaders import WebBaseLoader

# # TODO: replace with URLs from your domain
# urls = [
#     'https://en.wikipedia.org/wiki/YOUR_TOPIC_1',
#     'https://en.wikipedia.org/wiki/YOUR_TOPIC_2',
#     'https://docs.yourtool.io/getting-started',
# ]

# all_docs = WebBaseLoader(urls).load()
# all_text_chunks_raw = [doc.page_content for doc in all_docs]
# source_labels = [doc.metadata.get('source', url) for doc, url in zip(all_docs, urls)]
# print(f'✅ Loaded {len(all_text_chunks_raw)} pages from {len(urls)} URLs')

print('Option C is commented out. Uncomment the block above to load from URLs.')

---
## Step 2 — Chunk & Embed

**Your decision:** What chunk size should you use?

- **Smaller chunks (200–400 chars):** More precise retrieval, but individual chunks lose surrounding context.
- **Larger chunks (800–1200 chars):** More context per chunk, but retrieved chunks may be less focused on the query.

The right size depends on your documents. Dense technical docs → smaller chunks. Narrative text → larger chunks.

Change `CHUNK_SIZE` and `CHUNK_OVERLAP` below, then justify your choice in the comment.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# TODO: choose your chunk size and explain why in this comment
# My domain is [X] so I chose [Y] chars because [Z]
CHUNK_SIZE    = 400   # characters
CHUNK_OVERLAP = 80    # character overlap between chunks

# Chunk
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
docs  = [Document(page_content=t) for t in all_text_chunks_raw]
chunks = splitter.split_documents(docs)
all_chunks = [c.page_content for c in chunks]
print(f'Chunks: {len(all_chunks)} (from {len(all_text_chunks_raw)} source documents)')

# Embed + store
print('Loading embedding model (first run downloads ~46 MB)...')
embed_fn = SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
client_db = chromadb.Client()
collection = client_db.get_or_create_collection('capstone', embedding_function=embed_fn)
existing = collection.get().get('ids', [])
if existing:
    collection.delete(ids=existing)
collection.add(documents=all_chunks, ids=[str(i) for i in range(len(all_chunks))])
print(f'✅ Indexed {collection.count()} chunks into ChromaDB')

---
## Step 3 — Write Your Grounding Prompt

**This is the most important cell in the capstone.** The system prompt determines:
- Whether the model stays grounded to retrieved context or hallucinates
- What persona it adopts (expert, assistant, tutor)
- How it handles questions that aren't in the knowledge base

A weak system prompt: `'You are a helpful assistant.'`

A strong domain-specific prompt:
```
You are an expert assistant for [DOMAIN].
Answer ONLY based on the provided context.
If the context does not contain enough information, say exactly:
"I don't have information about that in my knowledge base."
Do not speculate or draw on outside knowledge.
Always cite the source topic at the end of your answer.
```

Write yours below — tailor it to your domain.

In [ ]:
from openai import OpenAI

oai = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)

# ── TODO: Write your domain-specific system prompt ────────────────────────
SYSTEM_PROMPT = '''
You are an expert assistant for [DESCRIBE YOUR DOMAIN HERE].
Answer ONLY based on the provided context.
If the context does not contain enough information, say:
"I don't have information about that in my knowledge base."
Do not speculate or use knowledge outside the provided context.

Context:
{context}
'''.strip()
# ─────────────────────────────────────────────────────────────────────────

def search(query, n=3):
    res = collection.query(query_texts=[query], n_results=n)
    return res['documents'][0]

def rag(question, n_chunks=3, stream=False):
    chunks = search(question, n=n_chunks)
    context = '\n\n---\n\n'.join(chunks)
    prompt  = SYSTEM_PROMPT.format(context=context)
    return oai.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=[{'role': 'system', 'content': prompt},
                  {'role': 'user',   'content': question}],
        stream=stream,
    )

print('✅ RAG function ready')

---
## Step 4 — Test Retrieval Quality

Before building the UI, verify that your pipeline is working:

1. Ask a question that **is** in your knowledge base — the answer should be grounded and accurate.
2. Ask a question that **is not** in your knowledge base — the model should say so, not hallucinate.
3. Ask a follow-up question that requires the **first question's answer** — watch what happens (stateless vs stateful).

If the retrieval isn't finding the right chunks, try adjusting `CHUNK_SIZE` in Step 2 and re-running.

In [ ]:
# ── Test 1: In-scope question ──────────────────────────────────────────────
# TODO: replace with a real question from your domain
q1 = 'YOUR IN-SCOPE QUESTION HERE'

print(f'Question: {q1}')
print('Retrieved chunks:')
for i, chunk in enumerate(search(q1), 1):
    print(f'  [{i}] {chunk[:100]}...')
print('\nAnswer:')
resp = rag(q1)
print(resp.choices[0].message.content)

In [ ]:
# ── Test 2: Out-of-scope question (should say 'I don't have info') ──────────
# TODO: replace with a question NOT in your knowledge base
q2 = 'YOUR OUT-OF-SCOPE QUESTION HERE'

print(f'Question: {q2}')
resp = rag(q2)
print(resp.choices[0].message.content)

---
## Step 5 — Build the App

The Gradio UI below is a complete working template from Lab 7. **You own two things:**

1. **`EXAMPLE_QUESTIONS`** — replace these with 3 real questions from your domain
2. **`APP_TITLE` and `APP_DESCRIPTION`** — describe your assistant to users

Everything else is functional as-is. Run the cell to get a public URL.

In [ ]:
import gradio as gr
import time

# ── TODO: customise for your domain ─────────────────────────────────────────
APP_TITLE       = 'My Domain RAG Assistant'     # TODO: give your app a name
APP_DESCRIPTION = 'Ask me about [YOUR DOMAIN].' # TODO: describe your knowledge base
EXAMPLE_QUESTIONS = [
    'YOUR EXAMPLE QUESTION 1',   # TODO: real question from your domain
    'YOUR EXAMPLE QUESTION 2',
    'YOUR EXAMPLE QUESTION 3',
]
# ─────────────────────────────────────────────────────────────────────────────

query_log = []

def respond(message, chat_history):
    start = time.time()
    chunks = search(message)
    context = '\n\n---\n\n'.join(chunks)
    prompt = SYSTEM_PROMPT.format(context=context)

    full_response = ''
    chat_history = chat_history + [[message, '']]
    for chunk in oai.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=[{'role': 'system', 'content': prompt},
                  {'role': 'user',   'content': message}],
        stream=True,
    ):
        delta = chunk.choices[0].delta.content or ''
        full_response += delta
        chat_history[-1][1] = full_response
        yield chat_history, '', ''

    latency = int((time.time() - start) * 1000)
    sources = '\n'.join(f'• {c[:80]}...' for c in chunks)
    query_log.append({'question': message, 'latency_ms': latency})
    yield chat_history, sources, f'Response time: {latency} ms'

with gr.Blocks(title=APP_TITLE) as demo:
    gr.Markdown(f'# {APP_TITLE}\n{APP_DESCRIPTION}')
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=400)
            msg     = gr.Textbox(placeholder='Ask a question...', label='Question')
            latency = gr.Textbox(label='', interactive=False)
        with gr.Column(scale=1):
            sources = gr.Textbox(label='Retrieved context', lines=12, interactive=False)

    gr.Examples(EXAMPLE_QUESTIONS, inputs=msg)
    msg.submit(respond, [msg, chatbot], [chatbot, sources, latency])

# Share generates a public URL valid for 72 hours
demo.launch(share=True)

---
## Step 6 — Evaluate (Bonus)

Run RAGAS to get objective metrics on your system's faithfulness and answer relevancy. Uses `gpt-4o` as judge — costs a few cents of API credits.

If RAGAS fails to install or throws errors, use the manual rubric below instead.

In [ ]:
# ── RAGAS automated evaluation ──────────────────────────────────────────────
# TODO: replace these with real questions from your domain
eval_questions = [
    'YOUR EVAL QUESTION 1',
    'YOUR EVAL QUESTION 2',
    'YOUR EVAL QUESTION 3',
]

try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy
    from ragas.llms import LangchainLLMWrapper
    from langchain_openai import ChatOpenAI
    from datasets import Dataset as HFDataset

    judge = LangchainLLMWrapper(ChatOpenAI(
        api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL, model='gpt-4o'))

    rows = []
    for q in eval_questions:
        ctxs = search(q)
        ans  = rag(q).choices[0].message.content
        rows.append({'question': q, 'answer': ans, 'contexts': ctxs, 'ground_truth': ''})

    ds = HFDataset.from_list(rows)
    result = evaluate(ds, metrics=[faithfulness, answer_relevancy],
                      llm=judge, raise_exceptions=False)
    print(result.to_pandas()[['question','faithfulness','answer_relevancy']].to_string())

except Exception as e:
    print(f'RAGAS unavailable ({e}). Use the manual rubric below.')
    print('\n── Manual Rubric ─────────────────────────────────────────────────────')
    for q in eval_questions:
        chunks = search(q)
        ans    = rag(q).choices[0].message.content
        print(f'\nQ: {q}')
        print(f'A: {ans[:200]}')
        print(f'Context used: {chunks[0][:100]}...')
        print('Rate: [1=hallucinating | 3=mostly grounded | 5=fully faithful]')

---
## Submission Checklist

Before your presentation, verify:

- [ ] My knowledge base covers a real domain (not the course example data)
- [ ] My system prompt is domain-specific (not just 'You are a helpful assistant')
- [ ] The Gradio app is live and has a public URL
- [ ] Example questions in the UI are from my domain
- [ ] I've tested at least one question the system **fails** on — I know why
- [ ] I can answer: what chunk size did I use and why?

**Presentation structure (5 minutes):**
1. *'I built a [domain] assistant for [who].'* (30 sec)
2. Architecture walkthrough: data → chunks → ChromaDB → retrieval → GPT-4o-mini → Gradio (60 sec)
3. Live demo: 2 good answers + 1 failure (2 min)
4. What surprised you / what you'd change (90 sec)